# 02 — Modelado Predictivo

## Objetivo

Construir y evaluar modelos predictivos que cuantifiquen la relación entre las materias primas y el costo operativo de los equipos.

Los resultados obtenidos permitirán:

- identificar las materias primas con mayor influencia sobre cada equipo;
- medir la capacidad predictiva de diferentes modelos;
- generar conocimiento reutilizable para el Asistente Inteligente de Planeación de Costos Operativos;
- servir como base para las herramientas de predicción utilizadas por el agente.

In [8]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append("../src")

from modeling import (
    split_data,
    train_linear_model,
    evaluate_model,
    get_feature_importance,
)

PROCESSED_PATH = Path("../data/processed")

## 1. Carga de Datos Procesados

In [9]:
df = pd.read_csv(PROCESSED_PATH / "historico_equipos_limpio.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.head()

,Date,Price_X,Price_Y,Price_Z,Price_Equipo1,Price_Equipo2
0,2010-01-04,80.12,527.5,2225.25,434.73,931.73
1,2010-01-05,80.59,527.5,2246.50,449.97,968.56
2,2010-01-06,81.89,527.5,2302.50,444.48,960.51
3,2010-01-07,81.51,527.5,2306.50,440.90,960.14
4,2010-01-08,81.37,552.5,2261.25,448.82,949.55


## 2.  Definición de Variables del Modelo

In [10]:
FEATURES = ["Price_X", "Price_Y", "Price_Z"]

TARGET_EQUIPO1 = "Price_Equipo1"
TARGET_EQUIPO2 = "Price_Equipo2"

## 3. Entrenamiento de Modelos — Equipo 1

In [11]:
from modeling import (
    time_series_split_data,
    train_linear_model,
    evaluate_model,
)

# División temporal
(
    X_train_eq1,
    X_test_eq1,
    y_train_eq1,
    y_test_eq1,
    train_df_eq1,
    test_df_eq1,
) = time_series_split_data(
    df=df,
    feature_cols=FEATURES,
    target_col=TARGET_EQUIPO1,
    test_size=0.2,
)

# Entrenamiento del modelo lineal
modelo_equipo1 = train_linear_model(
    X_train_eq1,
    y_train_eq1,
    model_type="linear",
)

# Evaluación sobre el conjunto de prueba
metricas_equipo1 = evaluate_model(
    modelo_equipo1,
    X_test_eq1,
    y_test_eq1,
)

metricas_equipo1

{'RMSE': np.float64(10.3088), 'MAE': 8.8699, 'R2': 0.9913}

### Hallazgo

El modelo de regresión lineal para el **Equipo 1** fue evaluado utilizando una partición temporal entre entrenamiento y prueba, preservando el orden cronológico de las observaciones.

Las métricas obtenidas constituyen la evidencia cuantitativa utilizada para valorar la capacidad predictiva del modelo y servirán como referencia para comparar modelos alternativos en las siguientes secciones.

> **Nota:** En esta etapa no se interpreta la influencia de las materias primas; dicho análisis se realiza posteriormente mediante la evaluación de la importancia de las variables.

El modelo fue evaluado mediante una partición temporal, utilizando aproximadamente el 80 % de las observaciones para entrenamiento y el 20 % restante para validación, preservando el orden cronológico de la serie.

In [13]:
# Desviación estándar de las variables predictoras
std_features = X_train_eq1.std()

# Importancia basada en coeficientes estandarizados
importancia_equipo1 = pd.DataFrame(
    {
        "Variable": FEATURES,
        "Coeficiente": modelo_equipo1.coef_,
    }
)

importancia_equipo1["Importancia"] = (
    importancia_equipo1["Coeficiente"]
    * std_features.values
).abs()

# Importancia relativa (%)
importancia_equipo1["Importancia_%"] = (
    importancia_equipo1["Importancia"]
    / importancia_equipo1["Importancia"].sum()
    * 100
)

# Ordenar por importancia
importancia_equipo1 = (
    importancia_equipo1
    .sort_values("Importancia", ascending=False)
    .reset_index(drop=True)
)

importancia_equipo1

,Variable,Coeficiente,Importancia,Importancia_%
0,Price_Y,0.796755,85.171548,93.640150
1,Price_X,0.203392,5.467229,6.010835
2,Price_Z,0.001188,0.317450,0.349014


### Hallazgo

La importancia relativa de las variables indica que **Price_Y** es, con amplia diferencia, la materia prima con mayor capacidad para explicar el comportamiento del **Equipo 1** dentro del modelo de regresión lineal.

De acuerdo con los coeficientes estandarizados, **Price_Y concentra aproximadamente el 93,64 % de la importancia relativa**, mientras que **Price_X** aporta cerca del **6,01 %** y **Price_Z** representa una contribución marginal del **0,35 %**.

Estos resultados evidencian que la capacidad predictiva del modelo depende principalmente de la información contenida en **Price_Y**, mientras que las demás materias primas aportan información complementaria.

**Clasificación del conocimiento:** Nivel A (Conocimiento crítico para el negocio).

**Uso posterior en el agente:** Este hallazgo permitirá responder preguntas como:

- ¿Cuál es la materia prima que más influye en el costo del Equipo 1?
- ¿Qué variable debería monitorearse con mayor prioridad para anticipar cambios en el precio del Equipo 1?
- ¿Cuál es la importancia relativa de cada materia prima dentro del modelo predictivo del Equipo 1?

## 4. Entrenamiento de Modelos — Equipo 2

In [14]:
from modeling import (
    time_series_split_data,
    train_linear_model,
    evaluate_model,
)

# División temporal
(
    X_train_eq2,
    X_test_eq2,
    y_train_eq2,
    y_test_eq2,
    train_df_eq2,
    test_df_eq2,
) = time_series_split_data(
    df=df,
    feature_cols=FEATURES,
    target_col=TARGET_EQUIPO2,
    test_size=0.2,
)

# Entrenamiento del modelo lineal
modelo_equipo2 = train_linear_model(
    X_train_eq2,
    y_train_eq2,
    model_type="linear",
)

# Evaluación sobre el conjunto de prueba
metricas_equipo2 = evaluate_model(
    modelo_equipo2,
    X_test_eq2,
    y_test_eq2,
)

metricas_equipo2

{'RMSE': np.float64(19.3198), 'MAE': 16.5514, 'R2': 0.9853}

### Hallazgo

El modelo de regresión lineal para el **Equipo 2** fue evaluado mediante una **partición temporal**, utilizando aproximadamente el **80 % de las observaciones para entrenamiento** y el **20 % restante para validación**, preservando el orden cronológico de la serie.

Las métricas obtenidas constituyen la evidencia utilizada para evaluar la capacidad predictiva del modelo y servirán como referencia para comparar su desempeño con otros modelos considerados en este proyecto.

> **Nota:** La influencia relativa de las materias primas sobre el **Equipo 2** se analizará en la siguiente sección mediante la evaluación de la importancia de las variables.

In [15]:
# Desviación estándar de las variables predictoras
std_features = X_train_eq2.std()

# Importancia basada en coeficientes estandarizados
importancia_equipo2 = pd.DataFrame(
    {
        "Variable": FEATURES,
        "Coeficiente": modelo_equipo2.coef_,
    }
)

importancia_equipo2["Importancia"] = (
    importancia_equipo2["Coeficiente"]
    * std_features.values
).abs()

# Importancia relativa (%)
importancia_equipo2["Importancia_%"] = (
    importancia_equipo2["Importancia"]
    / importancia_equipo2["Importancia"].sum()
    * 100
)

# Ordenar por importancia
importancia_equipo2 = (
    importancia_equipo2
    .sort_values("Importancia", ascending=False)
    .reset_index(drop=True)
)

importancia_equipo2

,Variable,Coeficiente,Importancia,Importancia_%
0,Price_Z,0.332109,88.722715,66.473832
1,Price_Y,0.332841,35.580095,26.657720
2,Price_X,0.341043,9.167328,6.868449


### Hallazgo

La importancia relativa calculada a partir de los coeficientes estandarizados muestra que **Price_Z** es la materia prima con mayor capacidad para explicar el comportamiento del **Equipo 2** dentro del modelo de regresión lineal.

Aunque los coeficientes originales presentan magnitudes similares, la estandarización permite realizar una comparación en una escala común, evidenciando que **Price_Z concentra la mayor parte de la capacidad explicativa del modelo**, mientras que **Price_Y** y **Price_X** presentan contribuciones considerablemente menores.

Este resultado identifica a **Price_Z** como la principal variable de interés para el seguimiento y la estimación del costo del **Equipo 2**.

**Clasificación del conocimiento:** Nivel A (Conocimiento crítico para el negocio).

**Uso posterior en el agente:** Este hallazgo permitirá responder preguntas como:

- ¿Qué materia prima influye más en el costo del Equipo 2?
- ¿Cuál variable debería monitorearse para anticipar cambios en el precio del Equipo 2?
- ¿Qué importancia relativa tiene cada materia prima dentro del modelo predictivo del Equipo 2?

## 5. Evaluación del Modelo Seleccionado

In [16]:
comparacion_modelos = pd.DataFrame(
    [
        {
            "Equipo": "Equipo 1",
            "Modelo": "Regresión Lineal",
            "RMSE": metricas_equipo1["RMSE"],
            "MAE": metricas_equipo1["MAE"],
            "R2": metricas_equipo1["R2"],
        },
        {
            "Equipo": "Equipo 2",
            "Modelo": "Regresión Lineal",
            "RMSE": metricas_equipo2["RMSE"],
            "MAE": metricas_equipo2["MAE"],
            "R2": metricas_equipo2["R2"],
        },
    ]
)

comparacion_modelos

,Equipo,Modelo,RMSE,MAE,R2
0,Equipo 1,Regresión Lineal,10.3088,8.8699,0.9913
1,Equipo 2,Regresión Lineal,19.3198,16.5514,0.9853


### Hallazgos

La evaluación del modelo de regresión lineal muestra un desempeño consistente para ambos equipos, con valores de **R² superiores al 98 %**, lo que evidencia una **elevada capacidad explicativa** sobre la variabilidad observada en los precios históricos.

Las métricas obtenidas indican que el modelo representa adecuadamente la relación entre las materias primas y el costo de los equipos dentro del período analizado, proporcionando una base sólida para identificar los principales drivers de costo.

### Conclusión

Considerando su elevada capacidad explicativa, su estabilidad y su facilidad de interpretación, la **regresión lineal** se adopta como **modelo explicativo de referencia** para analizar la influencia de las materias primas sobre el costo de los equipos.

Los hallazgos obtenidos en este notebook constituyen la base del conocimiento explicativo del proyecto y servirán como insumo para la etapa de forecasting desarrollada en el **Notebook 03**, donde se construirán modelos específicos para proyectar la evolución futura de los costos operativos.

**Clasificación del conocimiento:** Nivel A (Decisión metodológica del proyecto).

## 6. Conclusiones

El análisis desarrollado en este notebook permitió identificar las principales relaciones entre las materias primas y los costos históricos de los equipos, así como seleccionar el modelo explicativo que mejor representa dicho comportamiento.

Los hallazgos obtenidos muestran que:

- **Price_Y** constituye el principal driver del costo del **Equipo 1**.
- **Price_Z** constituye el principal driver del costo del **Equipo 2**.
- La regresión lineal ofrece una elevada capacidad explicativa para ambos equipos, manteniendo un equilibrio adecuado entre precisión e interpretabilidad.

Estos resultados proporcionan el fundamento explicativo del proyecto y constituyen la base para la siguiente etapa, enfocada en la construcción de modelos de series temporales que permitan proyectar el comportamiento futuro de los costos operativos.

En el **Notebook 03 — Forecasting**, el objetivo deja de ser explicar las relaciones históricas y pasa a estimar la evolución futura de los precios mediante modelos especializados para series temporales.